# Project 4 Live Demo：推論 + 控制（可切換模型）

流程：`camera.py` 取像 → `preprocessing.py`（把車道線標成紅色，和訓練時相同，固定開啟）→ `loader.py` 載入模型（`resnet18.py` / `mobilenet_v2.py`）並推論 → `control.py` 轉向角 + PID + 差速控制。畫面顯示的就是模型實際看到的影像。

- 從下拉選單選模型後，**載入完成就開始推論**，開不開車都會持續推論、顯示。
- **「開始行駛」按鈕**：按一下開始送馬達（按鈕變紅色「停止」），再按一下停車。安全機制停車時，按鈕會自動變回綠色，並顯示停車原因。
- 載入模型時會用 `.json` 裡存的畫面做 **sanity 比對**（TensorRT／FP16 轉換是否正確）；比對沒通過**禁止開車**，但推論照常顯示。
- 安全機制（開車時）：模型輸出 NaN／超出畫面、目標點太近、相機沒有新畫面、推論或相機逾時、推論出錯 → 立刻停車。
- 所有控制選項都是按鈕或滑桿，改了立刻生效。
- **沒有車也能開**：偵測不到 JetBot 就改用假馬達（只顯示數值）；鏡頭打不開就顯示「未連接鏡頭」。

下拉選單**只列 `models_trt/` 的 TensorRT 模型**。流程：把 PC 上 `train.py` 產生的 `.pth` + `.json`（要用 trtexec 的話加 `.onnx`）上傳到 `models/`，在車上執行 `python3 convert_trt.py` 轉換，結果放在 `models_trt/`。沒有 GPU 的電腦載入不了 TRT 模型，但介面照樣可以看。

## 1. 設定

In [ ]:
import os, sys, time, threading, csv, datetime, html, traceback
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display

# Make the modules in project4/ importable
sys.path.insert(0, os.path.abspath('.'))
# Camera sources, tensor conversion, input disturbance and drawing helpers
import camera as cam_mod
# Checkpoint loading (PyTorch / TensorRT) and listing for the dropdown
from loader import load_model, list_models
# Lane marking: the same preprocessing the models were trained with
from preprocessing import apply_frame as mark_lanes
# Steering controller and its default settings
from control import SteeringController, DEFAULTS as CTRL
# Robot-side comparison with the outputs stored at training time
import sanity

# Folder with the TensorRT models made by convert_trt.py (the only ones offered here)
MODELS_DIR = 'models_trt'
# Frame source: 'jetbot' | 'csi' (never queues old frames) | 'opencv' (webcam) | 'folder' | 'dummy'
CAMERA_SOURCE = 'jetbot'
# Image folder replayed when CAMERA_SOURCE = 'folder'
CAMERA_FOLDER = 'data/dataset_xy_2026-10-21/dataset_xy'
# Folder for the CSV drive logs
LOG_DIR = 'runs'
# Drop frames queued during a model switch (jetbot source restarts its pipeline, ~1-2 s)
FLUSH_ON_SWITCH = True
# Max display updates per second; higher rates make the Wi-Fi / browser fall behind
DISPLAY_FPS = 15

# Run on the GPU when there is one
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Show the device and the checkpoints that were found
print('device:', device, '| models:', [os.path.basename(p) for _, p in list_models(MODELS_DIR)])

## 2. 相機與車子（偵測不到就改用假的，沒有車也能看介面）

In [ ]:
# Stand-in for a jetbot motor when no robot is attached
class _FakeMotor(object):
    # Motor value written by the control loop
    value = 0.0

# Stand-in for jetbot.Robot with the same attributes the loop uses
class _FakeRobot(object):
    # Create two fake motors
    def __init__(self):
        # Left and right motor
        self.left_motor, self.right_motor = _FakeMotor(), _FakeMotor()
    # Set both motors to zero
    def stop(self):
        # Stop both motors
        self.left_motor.value = self.right_motor.value = 0.0

# Try the real robot first
try:
    from jetbot import Robot
    # Real motor driver
    robot = Robot()
    # Status text for the UI
    robot_text = '車子：已連接'
# No jetbot package or no motor driver: simulate it
except Exception as e:
    # Fake motors (values are only displayed)
    robot = _FakeRobot()
    # Status text with the reason
    robot_text = '車子：<b>未連接</b>（馬達值只顯示）— %s' % html.escape(repr(e)[:120])
# Make sure the robot starts standing still
robot.stop()

# Try to open the chosen frame source
try:
    # Camera object with read() / flush() / stop()
    camera = cam_mod.open_camera(CAMERA_SOURCE, folder=CAMERA_FOLDER)
    # Status text for the UI
    camera_text = '鏡頭：已連接（%s）' % CAMERA_SOURCE
# No camera (e.g. on a PC without jetbot): run without frames
except Exception as e:
    # No frames at all
    camera = None
    # Status text with the reason
    camera_text = '鏡頭：<b>未連接</b> — %s' % html.escape(repr(e)[:120])

# Controller with the default settings; the sliders overwrite them every frame
controller = SteeringController()
# Show what was found
print(robot_text.replace('<b>', '').replace('</b>', ''))
# Same for the camera
print(camera_text.replace('<b>', '').replace('</b>', ''))

## 3. 介面

In [ ]:
# Common layout for the parameter sliders
S = dict(layout=widgets.Layout(width='330px'), style={'description_width': '120px'})
# Common settings for the read-only value displays
R = dict(disabled=True, readout_format='.3f', **S)
# Common layout for the toggle buttons
T = dict(layout=widgets.Layout(width='160px'))

# --- Model ----------------------------------------------------------------
# Model selector filled with the checkpoints in MODELS_DIR; nothing selected at start
model_dd = widgets.Dropdown(description='模型', options=list_models(MODELS_DIR), value=None,
                            layout=widgets.Layout(width='520px'), style={'description_width': '50px'})
# Button to rescan MODELS_DIR
refresh_btn = widgets.Button(description='重新掃描', layout=widgets.Layout(width='90px'))
# Information line about the loaded model (and errors)
model_info = widgets.HTML(value='<b>未載入模型</b>：請從上面的下拉選單選擇')
# Result of the sanity comparison
sanity_info = widgets.HTML(value='')
# Robot and camera connection state
hw_info = widgets.HTML(value='%s ｜ %s' % (robot_text, camera_text))

# --- Picture --------------------------------------------------------------
# Live image (what the model sees, with the prediction drawn on it)
image_w = widgets.Image(format='jpeg', width=336, height=336)
# Placeholder shown instead of the picture when there is no camera
no_cam = widgets.HTML(value=('<div style="width:336px;height:336px;background:#333;color:#eee;display:flex;'
                             'align-items:center;justify-content:center;font-size:22px">未連接鏡頭</div>'))
# Show the placeholder only without a camera
(image_w if camera is None else no_cam).layout.display = 'none'

# --- Control settings (defaults from control.DEFAULTS) ---------------------
# Steering-angle formula
formula_tb = widgets.ToggleButtons(options=[('隊友公式', 'teammate'), ('投影片公式', 'slides')],
                                   value=CTRL['formula'], description='轉向角公式', style={'description_width': '120px'})
# Top speed on straights
max_speed_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=CTRL['max_speed'], description='max speed', **S)
# Lowest speed in sharp turns
min_speed_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=CTRL['min_speed'], description='min speed', **S)
# Proportional steering gain
kp_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=CTRL['kp'], description='kp', **S)
# Integral steering gain
ki_s = widgets.FloatSlider(min=0.0, max=0.5, step=0.005, value=CTRL['ki'], description='ki',
                           readout_format='.3f', **S)
# Derivative steering gain (per second)
kd_s = widgets.FloatSlider(min=0.0, max=0.1, step=0.001, value=CTRL['kd'], description='kd (每秒)',
                           readout_format='.3f', **S)
# Steering offset that compensates for unequal motors
bias_s = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=CTRL['bias'], description='steering bias', **S)
# How much the speed drops in sharp turns
slow_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=CTRL['turn_slowdown'], description='轉彎減速', **S)
# Low-pass filter on the angle (1 = off)
alpha_s = widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=CTRL['angle_alpha'], description='角度平滑 α', **S)
# Largest motor change per second (0 = off)
slew_s = widgets.FloatSlider(min=0.0, max=3.0, step=0.1, value=CTRL['slew_per_second'], description='馬達變化率/秒', **S)
# Largest motor value
maxout_s = widgets.FloatSlider(min=0.1, max=1.0, step=0.05, value=CTRL['max_output'], description='馬達上限', **S)
# Allow the inner wheel to spin backwards
reverse_tb = widgets.ToggleButton(description='內側輪可反轉', value=CTRL['allow_reverse'], **T)
# Stop when no successful control step happened for this long while driving
timeout_s = widgets.FloatSlider(min=0.2, max=2.0, step=0.1, value=0.5, description='安全逾時 (秒)', **S)

# --- Live readings --------------------------------------------------------
# Predicted x (normalised)
x_r = widgets.FloatSlider(min=-1.25, max=1.25, description='x', **R)
# Predicted y (normalised)
y_r = widgets.FloatSlider(min=-1.25, max=1.25, description='y', **R)
# Filtered steering angle in radians
angle_r = widgets.FloatSlider(min=-3.2, max=3.2, description='angle (rad)', **R)
# Total steering value
steer_r = widgets.FloatSlider(min=-1.0, max=1.0, description='steering', **R)
# Speed actually used
speed_r = widgets.FloatSlider(min=0.0, max=1.0, description='實際 speed', **R)
# Left motor value
left_r = widgets.FloatSlider(min=-1.0, max=1.0, description='left motor', **R)
# Right motor value
right_r = widgets.FloatSlider(min=-1.0, max=1.0, description='right motor', **R)
# FPS, latency and P / I / D terms
stats = widgets.HTML()

# --- Operation ------------------------------------------------------------
# Drive button: green "start" while standing, red "stop" while driving
drive_btn = widgets.Button(description='開始行駛', button_style='success', icon='play',
                           layout=widgets.Layout(width='160px', height='40px'))
# Driving state or the reason of the last stop
drive_status = widgets.HTML(value='停車中')
# Record every control step to a CSV file
record_tb = widgets.ToggleButton(description='錄製 CSV', value=False, icon='circle', **T)

# Arrange all widgets: model rows, picture + controls, operation row
ui = widgets.VBox([
    widgets.HBox([model_dd, refresh_btn]), model_info, sanity_info, hw_info,
    widgets.HBox([
        widgets.VBox([image_w, no_cam,
                      widgets.HBox([drive_btn, record_tb]), drive_status]),
        widgets.VBox([widgets.HTML('<b>控制參數</b>'), formula_tb, max_speed_s, min_speed_s, kp_s, ki_s, kd_s,
                      bias_s, slow_s, alpha_s, slew_s, maxout_s, reverse_tb, timeout_s]),
        widgets.VBox([widgets.HTML('<b>即時讀數</b>'), x_r, y_r, angle_r, steer_r, speed_r, left_r, right_r, stats]),
    ]),
])
# Show the interface
display(ui)

## 4. 推論 / 控制迴圈（執行後就在背景一直跑）

In [ ]:
# Shared state between the UI callbacks, the loop thread and the watchdog
state = {'model': None, 'pending': None, 'sanity': None, 'driving': False, 'last_ok': 0.0,
         'last_frame': None, 'frame_time': 0.0, 'pause': False, 'paused': False, 'closed': False,
         'csv': None, 'writer': None, 'loop_error': None}
# Lock around driving state changes (UI thread, loop thread, watchdog)
lock = threading.RLock()
# Readable texts for the controller's stop reasons
REASONS = {'invalid_prediction': '模型輸出 NaN / inf', 'out_of_range': '模型輸出超出畫面範圍',
           'target_too_close': '目標點太靠近車頭'}

# HTML summary of a loaded model for the info line
def describe(m):
    # Metadata from the checkpoint's .json
    meta = m.meta or {}
    # Start with file name, runtime and dtype
    parts = ['<b>%s</b>' % m.describe()]
    # Add the label (network | augmentation | compression)
    if meta.get('label'): parts.append(meta['label'])
    # Preprocessing the model was trained with
    pre = meta.get('preprocessing') or {}
    # Warn if the model was not trained on lane-marked images
    if pre.get('method') != 'lane_marking':
        # Red warning text
        parts.append('<span style="color:red">這個模型不是用車道線標示的資料訓練的</span>')
    # Parameter count
    if meta.get('params'): parts.append('params %.2fM' % (meta['params'] / 1e6))
    # Multiply-accumulate count
    if meta.get('macs'): parts.append('MACs %.2fG' % (meta['macs'] / 1e9))
    # Validation error: mean pixel distance if recorded, otherwise MSE
    if meta.get('val_metrics'): parts.append('val %.1f px' % meta['val_metrics']['mean_px'])
    # Older models only have the validation MSE
    elif meta.get('best_val_loss') is not None: parts.append('val loss %.4f' % meta['best_val_loss'])
    # Test error (only for models trained on split folders)
    if meta.get('test_metrics'): parts.append('test %.1f px (p95 %.1f)' % (meta['test_metrics']['mean_px'],
                                                                          meta['test_metrics']['p95_px']))
    # One line separated by bars
    return ' | '.join(parts)

# Show the drive button in its standing / driving look
def show_drive_button(driving):
    # Driving: red stop button
    if driving:
        # Red, "stop" text and icon
        drive_btn.description, drive_btn.button_style, drive_btn.icon = '停止', 'danger', 'stop'
    # Standing: green start button
    else:
        # Green, "start" text and icon
        drive_btn.description, drive_btn.button_style, drive_btn.icon = '開始行駛', 'success', 'play'

# Stop driving (button, safety check or model switch)
def stop_drive(reason=None, fault=False):
    # One state change at a time
    with lock:
        # No more motor commands
        state['driving'] = False
        # Stop both motors
        robot.stop()
    # Reset the motor readings
    left_r.value = right_r.value = 0.0
    # Back to the green start button
    show_drive_button(False)
    # Faults in red, normal stops in plain text
    drive_status.value = ('<span style="color:red"><b>已停車：%s</b></span>' % html.escape(reason) if fault
                          else '停車中' + ('：%s' % html.escape(reason) if reason else ''))

# Start driving if everything needed is there
def start_drive():
    # No frames, nothing to steer by
    if camera is None:
        # Explain
        drive_status.value = '<span style="color:red">未連接鏡頭，無法開車</span>'
        # Do not start
        return
    # No model loaded yet
    if state['model'] is None:
        # Explain
        drive_status.value = '<span style="color:red">未載入模型，無法開車</span>'
        # Do not start
        return
    # The sanity comparison failed: the converted model is not trustworthy
    if state['sanity'] and state['sanity']['status'] == 'failed':
        # Explain
        drive_status.value = '<span style="color:red">sanity 比對沒通過，禁止開車</span>'
        # Do not start
        return
    # One state change at a time
    with lock:
        # Fresh controller state: motors ramp up from 0
        controller.reset()
        # The watchdog counts from now
        state['last_ok'] = time.monotonic()
        # Motors may receive values
        state['driving'] = True
    # Red stop button
    show_drive_button(True)
    # Show the driving state
    drive_status.value = '<span style="color:red"><b>行駛中</b></span>'

# Drive button handler: toggle between start and stop
def on_drive(_=None):
    # Driving: stop
    if state['driving']:
        # Normal stop by the user
        stop_drive('手動停止')
    # Standing: start
    else:
        # Start if possible
        start_drive()

# Copy every control widget into the controller
def apply_controls():
    # Steering formula and speeds
    controller.formula, controller.max_speed, controller.min_speed = formula_tb.value, max_speed_s.value, min_speed_s.value
    # PID gains and bias
    controller.kp, controller.ki, controller.kd, controller.bias = kp_s.value, ki_s.value, kd_s.value, bias_s.value
    # Turn slowdown, angle filter and motor slew limit
    controller.turn_slowdown, controller.angle_alpha, controller.slew_per_second = slow_s.value, alpha_s.value, slew_s.value
    # Motor limit and reverse option
    controller.max_output, controller.allow_reverse = maxout_s.value, reverse_tb.value

# Open or close the CSV drive log
def set_recording(on):
    # Recording switched on and no file open yet
    if on and state['csv'] is None:
        # Create the log folder if needed
        os.makedirs(LOG_DIR, exist_ok=True)
        # File name with the current date and time
        path = os.path.join(LOG_DIR, 'drive_%s.csv' % datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
        # Open the file
        state['csv'] = open(path, 'w', newline='', encoding='utf-8')
        # CSV writer for it
        state['writer'] = csv.writer(state['csv'])
        # Header row
        state['writer'].writerow(['t', 'model', 'driving', 'x', 'y', 'raw_angle', 'angle',
                                  'p', 'i', 'd', 'steering', 'speed', 'left', 'right', 'status', 'latency_ms',
                                  'formula', 'kp', 'ki', 'kd', 'bias', 'min_speed', 'max_speed', 'turn_slowdown',
                                  'angle_alpha', 'slew_per_second', 'max_output', 'reverse'])
        # Show where the log goes
        drive_status.value += ' ｜ 錄製到 %s' % html.escape(path)
    # Recording switched off while a file is open
    elif not on and state['csv'] is not None:
        # Close the file
        state['csv'].close()
        # Forget the file and the writer
        state['csv'] = state['writer'] = None

# Load a model inside the loop thread, check it and make it the current one
def switch_model(path):
    # Never drive with a model that is being replaced
    stop_drive('切換模型')
    # Show that loading is in progress
    model_info.value = '載入中… %s' % html.escape(os.path.basename(path))
    # No model and no check result while loading
    state['model'], state['sanity'] = None, None
    # Clear the previous check result
    sanity_info.value = ''
    # Loading can fail (missing file, SHA256 mismatch, no TensorRT on a PC, ...)
    try:
        # Load the checkpoint
        m = load_model(path, device)
    # Show the problem and keep running without a model
    except Exception as e:
        # Red error text
        model_info.value = '<span style="color:red"><b>載入失敗：</b>%s</span>' % html.escape(repr(e))
        # No model
        return
    # Compare with the outputs stored at training time
    result = sanity.check(m)
    # Remember it for the drive button
    state['sanity'] = result
    # Colour by result: green passed, red failed, orange missing
    color = {'passed': 'green', 'failed': 'red'}.get(result['status'], 'darkorange')
    # Text by result
    word = {'passed': 'sanity 比對通過', 'failed': 'sanity 比對沒通過，禁止開車'}.get(result['status'], 'sanity 未比對')
    # Show the result
    sanity_info.value = '<span style="color:%s"><b>%s</b></span>：%s' % (color, word, html.escape(result['message']))
    # Warm-up: the first TRT / CUDA inferences are slow, so do them before the flush
    warm = camera.read() if camera is not None else None
    # Only possible once the camera has a frame
    if warm is not None:
        # A few warm-up runs
        for _ in range(3):
            # Same input path as the real loop
            m(cam_mod.preprocess(mark_lanes(warm), device, m.dtype))
    # Throw away frames queued during loading so the picture is live again
    if FLUSH_ON_SWITCH and camera is not None:
        # Show that the queue is being emptied
        model_info.value = '清空相機佇列… %s' % html.escape(os.path.basename(path))
        # Empty the camera queue
        camera.flush()
    # Start using the new model: inference begins right away
    state['model'] = m
    # Forget the previous controller state
    controller.reset()
    # Show the model summary
    model_info.value = describe(m)

# Inference + control loop, runs in a background thread for the whole session
def loop():
    # FPS measurement: window start, frames in window, last FPS
    fps_t, fps_n, fps = time.time(), 0, 0.0
    # Smoothed latency (None until the first frame)
    lat_avg = None
    # Time of the last display update
    disp_t = 0.0
    # Run until the end button
    while not state['closed']:
        # One iteration; an unexpected error must not kill the loop thread
        try:
            # The benchmark needs the GPU / camera alone
            if state['pause']:
                # Tell the benchmark the loop is idle
                state['paused'] = True
                # Wait
                time.sleep(0.05)
                # Check again
                continue
            # Not paused
            state['paused'] = False
            # A model was selected in the dropdown
            if state['pending'] is not None:
                # Take the request and clear it
                path, state['pending'] = state['pending'], None
                # Load, check and activate it
                switch_model(path)
                # Restart latency averaging and FPS measurement
                lat_avg, fps_t, fps_n, fps = None, time.time(), 0, 0.0
            # No camera: nothing to infer
            if camera is None:
                # Idle
                time.sleep(0.1)
                # Check again
                continue
            # Latest camera frame
            frame = camera.read()
            # Current time
            now = time.monotonic()
            # No frame yet, or the same frame as last time (camera has not delivered a new one)
            if frame is None or frame is state['last_frame']:
                # The camera stalled for too long while driving
                if state['driving'] and now - state['frame_time'] > timeout_s.value:
                    # Stop with the reason
                    stop_drive('相機超過 %.1f 秒沒有新畫面' % timeout_s.value, fault=True)
                # Wait a little
                time.sleep(0.005)
                # Try again
                continue
            # Remember the frame and when it arrived
            state['last_frame'], state['frame_time'] = frame, now
            # Start of the timed part (preprocessing + inference)
            t0 = time.time()
            # Preprocessing: paint the lane lines red (same as training)
            frame = mark_lanes(frame)
            # Current model
            m = state['model']
            # No model selected yet: just show the lane-marked picture
            if m is None:
                # Rate-limited display
                if time.time() - disp_t >= 1.0 / DISPLAY_FPS:
                    # Remember when it happened
                    disp_t = time.time()
                    # Picture without prediction
                    image_w.value = cam_mod.to_jpeg(frame)
                # Next frame
                continue
            # Inference can fail (e.g. CUDA / TensorRT errors)
            try:
                # Run the network and get the prediction as a flat float numpy array
                xy = m(cam_mod.preprocess(frame, device, m.dtype)).detach().float().cpu().numpy().flatten()
                # Wait for the GPU so the timing is correct
                if device.type == 'cuda':
                    # Block until all GPU work is done
                    torch.cuda.synchronize()
            # Stop, unload and show the error
            except Exception as e:
                # Stop driving
                stop_drive('推論錯誤：%r' % e, fault=True)
                # Unload the model; choose it again after fixing the problem
                state['model'] = None
                # Show the traceback
                model_info.value = ('<span style="color:red"><b>推論錯誤，模型已卸載：%s</b></span><pre style="font-size:11px">%s</pre>'
                                    % (html.escape(repr(e)), html.escape(traceback.format_exc())))
                # Next frame
                continue
            # Latency in milliseconds
            lat = (time.time() - t0) * 1000.0
            # Exponential moving average of the latency
            lat_avg = lat if lat_avg is None else 0.9 * lat_avg + 0.1 * lat

            # Take the current slider / button values
            apply_controls()
            # One control step
            left, right, status = controller.update(xy, time.monotonic())
            # All intermediate values of this step
            c = controller.last
            # Motor commands only while driving (checked and sent under the lock)
            with lock:
                # Driving state at this moment
                driving = state['driving']
                # Driving and the controller is happy: send the motor values
                if driving and status == 'ok':
                    # Send them
                    robot.left_motor.value, robot.right_motor.value = left, right
                    # The watchdog sees a successful step
                    state['last_ok'] = time.monotonic()
                # Standing: keep the motors at zero
                elif not driving:
                    # Stop both motors
                    robot.stop()
            # Driving but the controller refused (bad prediction / target too close): stop outside the lock
            if driving and status != 'ok':
                # Stop with the reason
                stop_drive(REASONS.get(status, status), fault=True)

            # Count this frame
            fps_n += 1
            # Update the FPS once per second
            if time.time() - fps_t >= 1.0:
                # Frames / elapsed time, then start a new window
                fps, fps_t, fps_n = fps_n / (time.time() - fps_t), time.time(), 0
            # Control runs every frame; the display at most DISPLAY_FPS times per second
            if time.time() - disp_t >= 1.0 / DISPLAY_FPS:
                # Remember when it happened
                disp_t = time.time()
                # Show x and y (clamped to the slider range)
                x_r.value, y_r.value = max(-1.25, min(1.25, float(xy[0]))), max(-1.25, min(1.25, float(xy[1])))
                # Show angle and (clamped) steering
                angle_r.value, steer_r.value = c['angle'], max(-1.0, min(1.0, c['steering']))
                # Show speed and the values actually sent (zero while standing)
                speed_r.value = c['speed']
                # Motor readings
                left_r.value, right_r.value = (left, right) if state['driving'] else (0.0, 0.0)
                # Lane-marked frame with prediction and steering bar
                image_w.value = cam_mod.to_jpeg(cam_mod.draw_overlay(frame, xy, c['steering']))
                # FPS, latency, PID terms and controller status
                stats.value = ('FPS <b>%.1f</b> | 前處理+推論 <b>%.1f ms</b>（平均 %.1f）<br>P %.3f I %.3f D %.3f | 狀態 %s'
                               % (fps, lat, lat_avg, c['p'], c['i'], c['d'], html.escape(REASONS.get(status, status))))
            # Open / close the log to follow the button
            set_recording(record_tb.value)
            # Write one log row while recording
            if state['writer'] is not None:
                # Time, model, driving flag, readings, status, latency and settings
                state['writer'].writerow(['%.3f' % time.time(), os.path.basename(m.path), int(state['driving'])]
                                         + ['%.4f' % c[k] for k in ('x', 'y', 'raw_angle', 'angle', 'p', 'i', 'd',
                                                                    'steering', 'speed', 'left', 'right')]
                                         + [status, '%.2f' % lat, controller.formula, controller.kp, controller.ki,
                                            controller.kd, controller.bias, controller.min_speed, controller.max_speed,
                                            controller.turn_slowdown, controller.angle_alpha,
                                            controller.slew_per_second, controller.max_output,
                                            int(controller.allow_reverse)])
        # Report the error, stop the robot and keep the loop alive
        except Exception as e:
            # Keep it for inspection
            state['loop_error'] = traceback.format_exc()
            # Never keep driving after an unexpected error
            stop_drive('迴圈錯誤：%r' % e, fault=True)
            # Show the traceback under the model selector
            model_info.value = ('<span style="color:red"><b>迴圈錯誤：%s</b></span><pre style="font-size:11px">%s</pre>'
                                % (html.escape(repr(e)), html.escape(state['loop_error'])))
            # Do not spin on a repeating error
            time.sleep(0.5)

    # Leave the robot in a safe state when the loop ends
    stop_drive('迴圈結束')
    # Close the log
    set_recording(False)

# Stops the robot when the loop or the camera stalls while driving
def watchdog():
    # Run until the end button
    while not state['closed']:
        # Check 20 times per second
        time.sleep(0.05)
        # Read the state consistently
        with lock:
            # Driving and no successful control step for too long
            late = state['driving'] and time.monotonic() - state['last_ok'] > timeout_s.value
        # Stop with the reason
        if late:
            # Fault stop
            stop_drive('推論或相機逾時（超過 %.1f 秒）' % timeout_s.value, fault=True)

# Dropdown change handler: load the chosen model
def on_model_change(change):
    # A model was selected
    if change['new']:
        # Request it; the loop thread loads it
        state['pending'] = change['new']

# "Rescan" button handler
def on_refresh(_=None):
    # Keep the current selection if it still exists
    current = model_dd.value
    # Re-read the checkpoints in MODELS_DIR
    model_dd.options = list_models(MODELS_DIR)
    # Restore the selection, or none
    model_dd.value = current if current in [p for _, p in model_dd.options] else None

# Connect the drive button
drive_btn.on_click(on_drive)
# Connect the rescan button
refresh_btn.on_click(on_refresh)
# React to dropdown changes
model_dd.observe(on_model_change, names='value')
# Start the loop thread (inference starts as soon as a model is loaded)
threading.Thread(target=loop, daemon=True).start()
# Start the watchdog thread
threading.Thread(target=watchdog, daemon=True).start()

## 5. 推論速度比較（報告用）

在**車子不動**的情況下，對 `models_trt/` 內每個模型用同一張畫面量 latency，條件相同才能比較（P.80）。執行時會先停車並暫停推論迴圈，量完自動恢復。沒有鏡頭時用一張灰色畫面。

In [ ]:
# Measure the inference latency of every checkpoint on the same frame (robot standing still)
def benchmark(n=100, warmup=20):
    # Never drive while benchmarking
    stop_drive('benchmark')
    # Ask the loop to pause
    state['pause'] = True
    # Wait until it is idle (at most 10 s)
    t_wait = time.time()
    # Poll the loop's idle flag
    while not state['paused']:
        # The loop thread did not answer
        if time.time() - t_wait > 10:
            # Resume and report
            state['pause'] = False
            # Explain
            raise RuntimeError('推論迴圈沒有回應，請 Restart Kernel')
        # Short wait
        time.sleep(0.02)
    # Always resume the loop afterwards
    try:
        # One frame shared by all models (grey frame without a camera)
        frame = camera.read() if camera is not None else None
        # Fallback frame
        if frame is None:
            # Uniform grey 224x224 BGR frame
            frame = np.full((224, 224, 3), 128, np.uint8)
        # (latency, label, runtime) per model
        rows = []
        # Every checkpoint in MODELS_DIR
        for label, path in list_models(MODELS_DIR):
            # Load it
            m = load_model(path, device)
            # Same input as in the loop (lane-marked)
            x = cam_mod.preprocess(mark_lanes(frame), device, m.dtype)
            # Warm-up runs, not timed
            for _ in range(warmup):
                # Run the network
                m(x)
            # Make sure the warm-up finished on the GPU
            if device.type == 'cuda': torch.cuda.synchronize()
            # Start the timer
            t0 = time.time()
            # Timed runs
            for _ in range(n):
                # Run the network
                m(x)
            # Wait for the GPU before stopping the timer
            if device.type == 'cuda': torch.cuda.synchronize()
            # Mean milliseconds per inference
            ms = (time.time() - t0) * 1000.0 / n
            # Store the result
            rows.append((ms, label, m.kind))
            # Free the model before loading the next one
            del m
    # Resume the loop
    finally:
        # Clear the pause request
        state['pause'] = False
    # Print fastest first
    for ms, label, kind in sorted(rows):
        # Latency, FPS, runtime and label
        print('%8.2f ms  %6.1f FPS  [%s]  %s' % (ms, 1000.0 / ms, kind, label))
    # Results for further use
    return rows

# Uncomment the next line to run the benchmark
# benchmark()

## 6. 結束：停車並釋放相機

做成按鈕，**只有手動按下才會執行**。如果直接寫成 cell，Run All 會一路跑到這裡，相機剛開就被關掉。

In [ ]:
# Button that stops everything and releases the camera
end_btn = widgets.Button(description='結束：停車並釋放相機', button_style='warning',
                         layout=widgets.Layout(width='220px'))

# "End" button handler
def on_end(_=None):
    # Stop the robot
    stop_drive('結束')
    # Ask the loop and the watchdog to end
    state['closed'] = True
    # Give the threads time to finish
    time.sleep(0.3)
    # Release the camera for other notebooks
    if camera is not None:
        # Stop the camera
        camera.stop()
    # No more driving
    drive_btn.disabled = True
    # The button can only be used once
    end_btn.disabled = True
    # Show that the camera is free
    end_btn.description = '相機已釋放'

# Connect the end button
end_btn.on_click(on_end)
# Show it
display(end_btn)